# AIkenGPT MMLU-style Evaluation — Letter scoring

このNotebookはOpenAI評価と同じ `MMLUEvaluator` を使います。
通常は次の **User settings** だけを変更し、GPU runtimeでRun allしてください。

処理順: Settings → Environment setup → Model loading → Dataset / evaluator setup
→ Preflight / manifest → Evaluation → Results


## 1. Settings

### User settings — 通常はこのセルだけを変更

In [31]:
from pathlib import Path

PROJECT_DIR = Path("/content/AIkenSGTv1_New")

# Choose where eval_data/ is loaded from: github, drive, or local.
DATA_SOURCE = "github"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/easy-jmmlu/eval_data")
LOCAL_DATA_DIR = Path("/content/eval_data")
DATA_REPO_DIR = Path("/content/AIkenSGTv1_easy_jmmlu")

# Evaluation repository
if not (PROJECT_DIR / "mmlu_eval").is_dir():
    !rm -rf /content/AIkenSGTv1_New
    !git clone https://github.com/HayatoHongo/AIkenSGTv1.git /content/AIkenSGTv1_New
    !cd /content/AIkenSGTv1_New && git switch tayama

# Dataset repository (separate branch from the evaluation code)
if DATA_SOURCE == "github":
    if not DATA_REPO_DIR.exists():
        !git clone --depth 1 --branch easy-jmmlu https://github.com/HayatoHongo/AIkenSGTv1.git /content/AIkenSGTv1_easy_jmmlu
    DATA_DIR = DATA_REPO_DIR / "easy-jmmlu" / "eval_data"
elif DATA_SOURCE == "drive":
    DATA_DIR = DRIVE_DATA_DIR
elif DATA_SOURCE == "local":
    DATA_DIR = LOCAL_DATA_DIR
else:
    raise ValueError("DATA_SOURCE must be github, drive, or local")


# 正解ラベルBの設問だけを評価用の一時データセットに抽出する。元データは変更しない。
import csv
import shutil

B_ONLY_DATA_DIR = Path("/content/jcommonsenseqa_gold_b_only")
if B_ONLY_DATA_DIR.exists():
    shutil.rmtree(B_ONLY_DATA_DIR)
shutil.copytree(DATA_DIR / "dev", B_ONLY_DATA_DIR / "dev")
(B_ONLY_DATA_DIR / "test").mkdir(parents=True)
B_ONLY_COUNT = 0
for source_csv in sorted((DATA_DIR / "test").glob("*.csv")):
    with source_csv.open(encoding="utf-8", newline="") as source:
        rows = list(csv.reader(source))
    rows = [row for row in rows if row and row[-1].strip().upper() == "B"]
    target_csv = B_ONLY_DATA_DIR / "test" / source_csv.name
    with target_csv.open("w", encoding="utf-8", newline="") as target:
        writer = csv.writer(target, lineterminator="\n")
        writer.writerows(rows)
    B_ONLY_COUNT += len(rows)
if B_ONLY_COUNT == 0:
    raise ValueError("No test questions with answer label B were found")
DATA_DIR = B_ONLY_DATA_DIR
print(f"正解Bの評価対象: {B_ONLY_COUNT}問")

OUTPUT_DIR = Path("/content/aikengpt_eval_output")

SUBJECT = "jcommonsenseqa"
NTRAIN = 0
SAMPLE_FRAC = 1.0
SEED = 42
LIMIT = 921
MANIFEST_PATH = None

### Project settings — 通常の問題セット変更では編集不要

In [33]:
MODEL_REPO = "HayatoHongo/AIkenSGTv1"
MODEL_REVISION = None
MODEL_PATH = None
MODEL_ID = "HayatoHongo/AIkenSGTv1:prompt_mask_instruction_tuned_model_epoch_1_lr_1e-04_easy_jmmlu.safetensors"
TOKENIZER = "gpt2"

DEVICE = "cuda"
DTYPE = "float32"
BATCH_SIZE = 1
MAX_CONTEXT_LENGTH = 2048

CONTEXT_POLICY = "reduce"
CONTEXT_TOKENIZER = "gpt2"

PERMUTATION_COUNT = 4
PERMUTATION_SEED = 0

SCORING_METHOD = "letter"
OUTPUT_NAME = f"aikengpt_{NTRAIN}shot_letter_jcommonsenseqa_answerB_only.csv"

## 2. Environment setup

In [34]:
%pip install -q numpy pandas tiktoken safetensors huggingface_hub


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, str(PROJECT_DIR))

assert (PROJECT_DIR / "mmlu_eval").is_dir(), (
    "PROJECT_DIR must point to the repository containing mmlu_eval/"
)
assert (DATA_DIR / "dev").is_dir(), "Missing dev folder under DATA_DIR: " + str(DATA_DIR)
assert (DATA_DIR / "test").is_dir(), "Missing test folder under DATA_DIR: " + str(DATA_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Model loading

In [ ]:
import torch
import tiktoken
from huggingface_hub import hf_hub_download
from mmlu_eval.backends.aikengpt_backend import (
    AIkenGPTBackend,
    file_sha256,
    load_custom_checkpoint,
)

assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a Colab GPU runtime"
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if MODEL_PATH is None:
    MODEL_PATH = Path(hf_hub_download(
        repo_id=MODEL_REPO,
        filename="model_clean.safetensors",
        revision=MODEL_REVISION,
    ))
else:
    MODEL_PATH = Path(MODEL_PATH)

checkpoint_hash = file_sha256(MODEL_PATH)
tokenizer = tiktoken.get_encoding(TOKENIZER)
model = load_custom_checkpoint(
    MODEL_PATH,
    device=DEVICE,
    dtype=DTYPE,
    max_context_length=MAX_CONTEXT_LENGTH,
)
print("Checkpoint SHA256:", checkpoint_hash)


GPU: NVIDIA A100-SXM4-80GB
Checkpoint SHA256: c6f8693f1446f435fb34ba8f621516221fc6cddd0c82326c7a00e37633bfd473


## 4. Dataset / evaluator setup

In [ ]:
from mmlu_eval import EvalConfig, MMLUEvaluator

eval_config = EvalConfig(
    ntrain=NTRAIN,
    sample_frac=SAMPLE_FRAC,
    seed=SEED,
    limit=LIMIT,
    subject=SUBJECT,
    permutation_count=PERMUTATION_COUNT,
    permutation_seed=PERMUTATION_SEED,
    context_policy=CONTEXT_POLICY,
    context_tokenizer=CONTEXT_TOKENIZER,
    max_context_length=MAX_CONTEXT_LENGTH,
)
evaluator = MMLUEvaluator(DATA_DIR, eval_config)


## 5. Preflight / manifest

In [ ]:
from mmlu_eval.core import atomic_csv

# Existing manifest is authoritative. Otherwise create one once and reuse it below.
manifest = evaluator.manifest(MANIFEST_PATH)
preflight_manifest_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.manifest.csv")
preflight_prompts_path = OUTPUT_DIR / (Path(OUTPUT_NAME).stem + ".preflight.prompts.jsonl")
atomic_csv(preflight_manifest_path, manifest)
evaluator.export_debug(manifest, preflight_prompts_path)
active_manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH is not None else preflight_manifest_path

first = manifest.iloc[0]
first_case = evaluator.cases(first.subject, int(first.test_index))[0]
print(f"Questions: {len(manifest)} / prompts: {len(manifest) * 4}")
print("Manifest:", active_manifest_path)
print("\nFirst prompt:\n")
print(first_case.prompt)


Questions: 30 / prompts: 120
Manifest: /content/drive/MyDrive/aikengpt_results/aikengpt_0shot_letter_easy_jmmlu.preflight.manifest.csv

First prompt:

The following are multiple choice questions (with answers) about jcommonsenseqa.

ビルや家などをまとめてなんという？
A. 木材
B. 葉っぱ
C. 鉄筋
D. 建物
Answer:


## 6. Evaluation

In [ ]:
backend = AIkenGPTBackend(
    model,
    tokenizer,
    model_identifier=MODEL_ID,
    tokenizer_identifier=TOKENIZER,
    scoring_method=SCORING_METHOD,
    text_reduction=None,
    device=DEVICE,
    dtype=DTYPE,
    batch_size=BATCH_SIZE,
    max_context_length=MAX_CONTEXT_LENGTH,
    checkpoint_sha256=checkpoint_hash,
)

OUTPUT_PATH = OUTPUT_DIR / OUTPUT_NAME
results = evaluator.run(
    backend,
    OUTPUT_PATH,
    manifest_path=active_manifest_path,
)


ValueError: Resume settings/model/manifest mismatch or legacy CSV; use a new output path

## 7. Results

In [ ]:
from mmlu_eval.core import summarize

print(summarize(results))
print("Result CSV:", OUTPUT_PATH)
print("Manifest:", OUTPUT_PATH.with_suffix(".manifest.csv"))
print("Prompts:", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
display(results.head())

# モデルの回答（選択肢）の件数と割合
answer_counts = results["debiased_pred"].value_counts().reindex(list("ABCD"), fill_value=0)
answer_distribution = answer_counts.to_frame(name="件数")
answer_distribution["割合"] = (answer_counts / len(results)).map(lambda x: f"{x:.1%}")
display(answer_distribution)


# OpenAI runとの入力一致を確認する場合:
# from mmlu_eval.compare import compare
# compare("/path/to/openai.prompts.jsonl", OUTPUT_PATH.with_suffix(".prompts.jsonl"))
